# 3.2 Build meta-features

**Input:** validation component predictions.

**Output:** one Parquet file containing the 19 inputs used by each residual stacker.

In [1]:
# Step 1 - Imports and inputs

import os
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

HORIZONS = range(1, variables.HORIZON_COUNT+1)
META_FEATURES = [
    "base_l1", "base_l2", "spike_mae", "spike_q90", "dip_mae", "dip_q10",
    "spike_probability", "dip_probability", "spike_logit", "dip_logit",
    "base_l2_minus_l1", "spike_mae_positive_gap", "spike_q90_positive_gap",
    "dip_mae_negative_gap", "dip_q10_negative_gap", "weighted_spike_mae_gap",
    "weighted_spike_q90_gap", "weighted_dip_mae_gap", "weighted_dip_q10_gap",
]
component_predictions = pd.read_parquet(variables.VALIDATION_COMPONENT_PREDICTIONS_PATH)

In [2]:
# Step 2 - Build the 19 features

def build_meta_features(horizon):
    prediction = {name: component_predictions[f"{name}_h{horizon}"].values for name in ["base_l1", "base_l2", "spike_probability", "spike_mae", "spike_q90", "dip_probability", "dip_mae", "dip_q10"]}
    base_l1 = prediction["base_l1"]
    spike_probability = np.clip(prediction["spike_probability"], 1e-6, 1-1e-6)
    dip_probability = np.clip(prediction["dip_probability"], 1e-6, 1-1e-6)
    spike_mae_gap = np.maximum(prediction["spike_mae"]-base_l1, 0)
    spike_q90_gap = np.maximum(prediction["spike_q90"]-base_l1, 0)
    dip_mae_gap = np.minimum(prediction["dip_mae"]-base_l1, 0)
    dip_q10_gap = np.minimum(prediction["dip_q10"]-base_l1, 0)

    values = {
        **prediction,
        "spike_probability": spike_probability,
        "dip_probability": dip_probability,
        "spike_logit": np.log(spike_probability/(1-spike_probability)),
        "dip_logit": np.log(dip_probability/(1-dip_probability)),
        "base_l2_minus_l1": prediction["base_l2"]-base_l1,
        "spike_mae_positive_gap": spike_mae_gap,
        "spike_q90_positive_gap": spike_q90_gap,
        "dip_mae_negative_gap": dip_mae_gap,
        "dip_q10_negative_gap": dip_q10_gap,
        "weighted_spike_mae_gap": spike_probability*spike_mae_gap,
        "weighted_spike_q90_gap": spike_probability*spike_q90_gap,
        "weighted_dip_mae_gap": dip_probability*dip_mae_gap,
        "weighted_dip_q10_gap": dip_probability*dip_q10_gap,
    }
    return {f"{name}_h{horizon}": values[name].astype(np.float32) for name in META_FEATURES}

In [3]:
# Step 3 - Save

meta_features = {}
for horizon in tqdm(HORIZONS):
    meta_features.update(build_meta_features(horizon))

meta_features = pd.DataFrame(meta_features, index=component_predictions.index)
meta_features.index.name = "date"
meta_features.to_parquet(variables.VALIDATION_META_FEATURES_PATH)

display(meta_features.head())
print(meta_features.shape)
print(variables.VALIDATION_META_FEATURES_PATH)

100%|██████████| 96/96 [00:00<00:00, 496.53it/s]


,base_l1_h1,base_l2_h1,spike_mae_h1,spike_q90_h1,dip_mae_h1,dip_q10_h1,spike_probability_h1,dip_probability_h1,spike_logit_h1,dip_logit_h1,...,dip_logit_h96,base_l2_minus_l1_h96,spike_mae_positive_gap_h96,spike_q90_positive_gap_h96,dip_mae_negative_gap_h96,dip_q10_negative_gap_h96,weighted_spike_mae_gap_h96,weighted_spike_q90_gap_h96,weighted_dip_mae_gap_h96,weighted_dip_q10_gap_h96
date,,,,,,,,,,,,,,,,,,,,,
2024-07-01 00:05:00,64.090195,61.883320,61.385937,71.683853,66.089874,60.913990,0.001023,0.000048,-6.883805,-9.937917,...,-5.164676,25.978661,35.338943,102.342773,-16.363564,-39.082207,5.221331,15.121150,-0.092985,-0.222082
2024-07-01 00:10:00,78.125633,79.173218,83.646301,123.826744,77.907288,63.872150,0.002832,0.000061,-5.863791,-9.707420,...,-5.164676,22.161880,46.883553,111.671227,-21.095078,-42.855034,10.058911,23.959169,-0.119871,-0.243521
2024-07-01 00:15:00,83.100830,85.290306,84.653915,124.099098,84.113670,68.667892,0.003504,0.000055,-5.650321,-9.811193,...,-5.164676,20.000435,49.636147,114.134117,-19.168716,-40.928673,11.038023,25.381001,-0.108925,-0.232575
2024-07-01 00:20:00,87.827690,86.080711,88.036880,123.253586,87.431595,68.881271,0.003171,0.000050,-5.750678,-9.909071,...,-5.164676,15.154564,45.219177,106.160187,-24.137993,-46.856636,9.910308,23.266237,-0.137163,-0.266260
2024-07-01 00:25:00,91.239716,89.347298,94.064323,122.519569,87.947021,69.393562,0.003956,0.000044,-5.528495,-10.026891,...,-5.164676,5.913887,32.639084,96.156265,-37.595169,-59.645668,6.819164,20.089577,-0.213632,-0.338933


(52992, 1824)
/home/daniel-davaris/Documents/NEM-Short-Term-Price-Forecasting/EPF/5_Model/Data/4_combine_models/2_validation_meta_features.parquet
